# DATAENG_300 Homework 3
## Nick Visuthikosol

## Part 1: EC2 and Spark setup

In [32]:
import subprocess
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)
import pyspark
print(pyspark.__version__)

openjdk version "17.0.10" 2024-01-16 LTS
OpenJDK Runtime Environment Corretto-17.0.10.7.1 (build 17.0.10+7-LTS)
OpenJDK 64-Bit Server VM Corretto-17.0.10.7.1 (build 17.0.10+7-LTS, mixed mode, sharing)

4.0.2


## Part 2: Start Spark

In [33]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC Taxi Analytics Assignment")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

## Part 3: Load yellow and green taxi data

In [34]:
yellow_path = "/home/ec2-user/data/nyctlc/yellow_tripdata_2026-01.parquet"
green_path  = "/home/ec2-user/data/nyctlc/green_tripdata_2026-01.parquet"

yellow_raw = spark.read.parquet(yellow_path)
green_raw  = spark.read.parquet(green_path)

# inspect schemas
yellow_raw.printSchema()
green_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: ti

In [35]:
yellow_raw.show(5, truncate=False)
green_raw.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2026-01-01 00:54:04 |2026-01-01 00:59:37  |1              |0.97         |1         |N                 |239         |238 

## Part 4: Standardize schemas

In [36]:
from pyspark.sql.functions import col, lit

# yellow taxi: rename tpep columns to standard names
yellow = (
    yellow_raw
    .withColumnRenamed("tpep_pickup_datetime",  "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    .withColumnRenamed("PULocationID",          "pickup_location_id")
    .withColumnRenamed("DOLocationID",          "dropoff_location_id")
    .withColumn("taxi_type", lit("yellow"))
)

In [37]:
# green taxi: rename lpep columns to standard names
green = (
    green_raw
    .withColumnRenamed("lpep_pickup_datetime",  "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    .withColumnRenamed("PULocationID",          "pickup_location_id")
    .withColumnRenamed("DOLocationID",          "dropoff_location_id")
    .withColumn("taxi_type", lit("green"))
)

In [38]:
common_cols = [
    "taxi_type",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_location_id",
    "dropoff_location_id",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "passenger_count",
]

yellow = yellow.select(common_cols)
green  = green.select(common_cols)
df = yellow.unionByName(green)

print(f"Total rows: {df.count()}")
df.printSchema()

Total rows: 3765161
root
 |-- taxi_type: string (nullable = false)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- passenger_count: long (nullable = true)



## Part 5: Clean the data

In [39]:
from pyspark.sql.functions import col, unix_timestamp

df_clean = (
    df
    #missing pickup or dropoff timestamps
    .filter(col("pickup_datetime").isNotNull())
    .filter(col("dropoff_datetime").isNotNull())
    # non-positive distance
    .filter(col("trip_distance") > 0)
    # negative fares
    .filter(col("fare_amount") >= 0)
    # negative total amount
    .filter(col("total_amount") >= 0)
    # trips where dropoff is before pickup
    .filter(col("dropoff_datetime") > col("pickup_datetime"))
    # trips longer than 24 hours
    .filter((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) <= 86400)
)

print(f"Rows before cleaning: {df.count()}")
print(f"Rows after cleaning:  {df_clean.count()}")

Rows before cleaning: 3765161
Rows after cleaning:  3557037


## Part 6: Required analytics

### Question 1: Which taxi type had more trips?

In [40]:
trips_by_type = (
    df_clean
    .groupBy("taxi_type")
    .count()
    .orderBy("count", ascending=False)
)

trips_by_type.show()

[Stage 129:======================================>                  (2 + 1) / 3]

+---------+-------+
|taxi_type|  count|
+---------+-------+
|   yellow|3518128|
|    green|  38909|
+---------+-------+



### Question 2: What was the average fare by taxi type?


In [41]:
from pyspark.sql.functions import avg, round as spark_round

avg_fare_by_type = (
    df_clean
    .groupBy("taxi_type")
    .agg(spark_round(avg("fare_amount"), 2).alias("avg_fare"))
    .orderBy("taxi_type")
)

avg_fare_by_type.show()

+---------+--------+
|taxi_type|avg_fare|
+---------+--------+
|    green|    16.0|
|   yellow|   21.08|
+---------+--------+



### Question 3: What was the average trip distance by taxi type?

In [42]:
avg_distance_by_type = (
    df_clean
    .groupBy("taxi_type")
    .agg(spark_round(avg("trip_distance"), 2).alias("avg_distance"))
    .orderBy("taxi_type")
)

avg_distance_by_type.show()

[Stage 135:>                                                        (0 + 2) / 3]

+---------+------------+
|taxi_type|avg_distance|
+---------+------------+
|    green|       12.99|
|   yellow|        6.76|
+---------+------------+



### Question 4: What hour of day had the most pickups?

In [43]:
from pyspark.sql.functions import hour

pickups_by_hour = (
    df_clean
    .withColumn("pickup_hour", hour("pickup_datetime"))
    .groupBy("pickup_hour")
    .count()
    .orderBy("count", ascending=False)
)

pickups_by_hour.show(24)

[Stage 138:======================================>                  (2 + 1) / 3]

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|232570|
|         17|224277|
|         15|216305|
|         19|211118|
|         20|208072|
|         21|207428|
|         16|206485|
|         14|202317|
|         13|190970|
|         22|190251|
|         12|183146|
|         11|167923|
|         10|158111|
|          9|154087|
|         23|148822|
|          8|146020|
|          7|112027|
|          0|109013|
|          1| 75623|
|          6| 62649|
|          2| 52470|
|          3| 37491|
|          5| 32070|
|          4| 27792|
+-----------+------+



### Question 5: What percentage of trips were under 2 miles?

In [44]:
total     = df_clean.count()
under_2   = df_clean.filter(col("trip_distance") < 2).count()
pct       = round((under_2 / total) * 100, 2)

print(f"Trips under 2 miles: {under_2}")
print(f"Total trips:         {total}")
print(f"Percentage:          {pct}%")

Trips under 2 miles: 1847954
Total trips:         3557037
Percentage:          51.95%


### Question 8: Predict the fare amount using non-fare predictor columns.

In [45]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [46]:
# non-fare columns as predictors
feature_cols = [
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "passenger_count",
]
# Drop rows with nulls 
df_model = df_clean.select(feature_cols + ["fare_amount"]).dropna()

In [47]:
# Train/test split
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows: {train_df.count()}")
print(f"Test rows:     {test_df.count()}")

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

Training rows: 2045084


[Stage 150:======================================>                  (2 + 1) / 3]

Test rows:     511300


In [48]:
# Random forest model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    numTrees=80,
    maxDepth=8,
    seed=42,
)

pipeline = Pipeline(stages=[assembler, rf])
model = pipeline.fit(train_df)
print("Model fitted.")

26/05/23 12:58:17 WARN MemoryStore: Not enough space to cache rdd_530_1 in memory! (computed 155.3 MiB so far)
26/05/23 12:58:17 WARN BlockManager: Persisting block rdd_530_1 to disk instead.
26/05/23 12:58:17 WARN MemoryStore: Not enough space to cache rdd_530_0 in memory! (computed 45.3 MiB so far)
26/05/23 12:58:17 WARN BlockManager: Persisting block rdd_530_0 to disk instead.
26/05/23 12:58:36 WARN MemoryStore: Not enough space to cache rdd_530_0 in memory! (computed 241.6 MiB so far)
26/05/23 12:58:50 WARN MemoryStore: Not enough space to cache rdd_530_0 in memory! (computed 241.6 MiB so far)
26/05/23 12:59:11 WARN MemoryStore: Not enough space to cache rdd_530_0 in memory! (computed 241.6 MiB so far)
26/05/23 12:59:37 WARN MemoryStore: Not enough space to cache rdd_530_0 in memory! (computed 241.6 MiB so far)
26/05/23 13:00:06 WARN MemoryStore: Not enough space to cache rdd_530_0 in memory! (computed 241.6 MiB so far)
26/05/23 13:00:42 WARN MemoryStore: Not enough space to cache 

Model fitted.


#### Evaluate the model

In [49]:
train_preds = model.transform(train_df)
test_preds  = model.transform(test_df)

evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

In [50]:
train_rmse = evaluator.evaluate(train_preds)
test_rmse  = evaluator.evaluate(test_preds)

print(f"Training RMSE: ${train_rmse:,.2f}")
print(f"Test RMSE:     ${test_rmse:,.2f}")

[Stage 174:======================================>                  (2 + 1) / 3]

Training RMSE: $7.39
Test RMSE:     $7.68


In [51]:
# Feature importances
import pandas as pd
rf_model    = model.stages[-1]
importances = rf_model.featureImportances.toArray()

importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": importances})
    .sort_values("importance", ascending=False)
)
print(importance_df)

               feature  importance
0        trip_distance    0.819885
1   pickup_location_id    0.130795
2  dropoff_location_id    0.045709
3      passenger_count    0.003611


#### Plot actual vs predicted

In [57]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plot_pdf = (
    test_preds
    .select("fare_amount", "prediction")
    .sample(fraction=0.25, seed=7)
    .limit(1000)
    .toPandas()
)

plt.figure(figsize=(8, 6))
plt.scatter(plot_pdf["fare_amount"], plot_pdf["prediction"], alpha=0.35)

min_val = min(plot_pdf["fare_amount"].min(), plot_pdf["prediction"].min())
max_val = max(plot_pdf["fare_amount"].max(), plot_pdf["prediction"].max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Actual Fare Amount")
plt.ylabel("Predicted Fare Amount")
plt.title("Random Forest: Actual vs Predicted Fare")
plt.tight_layout()
plt.savefig("/home/ec2-user/actual_vs_predicted.png")
plt.show()

###

In [58]:
import subprocess

# Save results locally first
trips_by_type.write.mode("overwrite").parquet("/home/ec2-user/outputs/trips_by_type")
avg_fare_by_type.write.mode("overwrite").parquet("/home/ec2-user/outputs/avg_fare_by_type")
pickups_by_hour.write.mode("overwrite").parquet("/home/ec2-user/outputs/pickups_by_hour")

output_base = "s3://de300-hw3-visuthikosol/nyc-taxi-assignment"
subprocess.run(["aws", "s3", "cp", "/home/ec2-user/outputs/", f"{output_base}/", "--recursive"])
subprocess.run(["aws", "s3", "cp", "/home/ec2-user/actual_vs_predicted.png", f"{output_base}/actual_vs_predicted.png"])

print("uploaded to S3")

upload: outputs/avg_fare_by_type/_SUCCESS to s3://de300-hw3-visuthikosol/nyc-taxi-assignment/avg_fare_by_type/_SUCCESS
upload: outputs/pickups_by_hour/.part-00000-37711e60-cbd5-436e-a058-c52124c074f5-c000.snappy.parquet.crc to s3://de300-hw3-visuthikosol/nyc-taxi-assignment/pickups_by_hour/.part-00000-37711e60-cbd5-436e-a058-c52124c074f5-c000.snappy.parquet.crc
upload: outputs/avg_fare_by_type/.part-00000-d92d3d9e-7f43-4df0-8621-643c47fbf409-c000.snappy.parquet.crc to s3://de300-hw3-visuthikosol/nyc-taxi-assignment/avg_fare_by_type/.part-00000-d92d3d9e-7f43-4df0-8621-643c47fbf409-c000.snappy.parquet.crc
upload: outputs/avg_fare_by_type/._SUCCESS.crc to s3://de300-hw3-visuthikosol/nyc-taxi-assignment/avg_fare_by_type/._SUCCESS.crc
upload: outputs/trips_by_type/part-00000-a6c09784-1156-4193-89ac-8ae302a2c8a2-c000.snappy.parquet to s3://de300-hw3-visuthikosol/nyc-taxi-assignment/trips_by_type/part-00000-a6c09784-1156-4193-89ac-8ae302a2c8a2-c000.snappy.parquet
upload: outputs/trips_by_type